# Urchin v1 YOLO26m training notebook

Train an Ultralytics YOLO detection model on the prepared urchin v1 split generated by the dataset review notebook. This notebook uses the split `data.yaml`, validates image-label parity before training, starts from `yolo26m.pt`, and writes all training outputs under the v1 folder.

Reference pattern: Ultralytics Python API (`YOLO(...).train(...)`, `val`, `predict`, `export`) from https://github.com/ultralytics/ultralytics.

In [ ]:
from __future__ import annotations

import json
import os
import random
import shutil
import subprocess
import sys
from collections import Counter
from datetime import datetime
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import Image as IPythonImage
from IPython.display import display

try:
    import torch
except ImportError as error:
    raise ImportError("Install PyTorch in this notebook environment before training.") from error

In [ ]:
PROJECT_ROOT = Path("c:/Users/michael.akridge/Desktop/20260713/optics-si-special-projects/urchins/multi-class/v1")
SPLIT_ROOT = PROJECT_ROOT / "dataset_review_and_split" / "split"
DATA_YAML = SPLIT_ROOT / "data.yaml"
TRAINING_ROOT = PROJECT_ROOT / "training_runs"
MODELS_DIR = PROJECT_ROOT / "models"

MODEL_NAME = "yolo26m.pt"
FALLBACK_MODEL_NAME = "yolo11m.pt"
ALLOW_MODEL_FALLBACK = True

RUN_NAME = "yolo26m_urchins_v1_seed_20260713"
SEED = 20260713
EPOCHS = 250
IMGSZ = 960
BATCH = 8
PATIENCE = 45
SAVE_PERIOD = 10
WORKERS = 4 if os.name == "nt" else 8
CACHE = False
EXIST_OK = True

OPTIMIZER = "AdamW"
LR0 = 5e-4
LRF = 0.01
WEIGHT_DECAY = 0.001
COS_LR = True
CLOSE_MOSAIC = 15
FREEZE = None

DEVICE = 0 if torch.cuda.is_available() else "cpu"
AMP = torch.cuda.is_available()
RUN_FULL_TRAINING = True
RUN_SMOKE_TEST = False

PREDICT_CONF = 0.25
PREDICT_MAX_DET = 300
EXPORT_ONNX = True

for path in [TRAINING_ROOT, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print({
    "project_root": str(PROJECT_ROOT),
    "data_yaml": str(DATA_YAML),
    "model_name": MODEL_NAME,
    "run_name": RUN_NAME,
    "device": DEVICE,
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
})

In [ ]:
def run_command(command: list[str]) -> str:
    try:
        completed = subprocess.run(command, capture_output=True, text=True, check=True)
        return completed.stdout.strip()
    except Exception as error:  # noqa: BLE001
        return f"{type(error).__name__}: {error}"

runtime_info = {
    "python_executable": sys.executable,
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_count": torch.cuda.device_count(),
    "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(runtime_info)

if torch.cuda.is_available():
    print("nvidia-smi:")
    print(run_command(["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.used", "--format=csv,noheader"]))
else:
    print("CUDA is not available. Training will run on CPU unless you switch kernels/environments.")

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def resolve_yaml_path(data_yaml_path: Path, value: str) -> Path:
    path = Path(value)
    return path if path.is_absolute() else data_yaml_path.parent / path

def read_data_yaml(data_yaml_path: Path) -> dict:
    if not data_yaml_path.exists():
        raise FileNotFoundError(f"Missing data.yaml: {data_yaml_path}")
    with data_yaml_path.open("r", encoding="utf-8") as file:
        config = yaml.safe_load(file)
    names = config.get("names")
    if isinstance(names, dict):
        names = [names[index] for index in sorted(names)]
    config["names"] = list(names)
    if int(config["nc"]) != len(config["names"]):
        raise ValueError(f"data.yaml nc={config['nc']} but names has {len(config['names'])} entries")
    return config

def parse_label_file(label_path: Path, nc: int) -> tuple[list[int], list[dict]]:
    class_ids = []
    issues = []
    for line_number, raw_line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) != 5:
            issues.append({"label": str(label_path), "line": line_number, "issue": "expected_5_fields", "value": line})
            continue
        try:
            class_id = int(float(parts[0]))
            coords = [float(value) for value in parts[1:]]
        except ValueError:
            issues.append({"label": str(label_path), "line": line_number, "issue": "non_numeric_value", "value": line})
            continue
        if class_id < 0 or class_id >= nc:
            issues.append({"label": str(label_path), "line": line_number, "issue": "class_id_out_of_range", "value": class_id})
        if any(coord < 0 or coord > 1 for coord in coords):
            issues.append({"label": str(label_path), "line": line_number, "issue": "bbox_coord_out_of_range", "value": coords})
        class_ids.append(class_id)
    return class_ids, issues

data_config = read_data_yaml(DATA_YAML)
class_names = data_config["names"]
nc = int(data_config["nc"])

split_rows = []
class_counter = Counter()
image_class_counter = Counter()
pair_issues = []
label_issues = []

for split in ["train", "val", "test"]:
    image_dir = resolve_yaml_path(DATA_YAML, data_config[split])
    label_dir = DATA_YAML.parent / "labels" / split
    if not image_dir.exists():
        pair_issues.append({"split": split, "issue": "missing_image_dir", "path": str(image_dir)})
        continue
    if not label_dir.exists():
        pair_issues.append({"split": split, "issue": "missing_label_dir", "path": str(label_dir)})
        continue

    image_paths = sorted(path for path in image_dir.iterdir() if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS)
    label_paths = sorted(label_dir.glob("*.txt"))
    image_stems = {path.stem for path in image_paths}
    label_stems = {path.stem for path in label_paths}
    missing_labels = sorted(image_stems - label_stems)
    orphan_labels = sorted(label_stems - image_stems)

    if missing_labels:
        pair_issues.append({"split": split, "issue": "missing_labels", "count": len(missing_labels), "examples": missing_labels[:5]})
    if orphan_labels:
        pair_issues.append({"split": split, "issue": "orphan_labels", "count": len(orphan_labels), "examples": orphan_labels[:5]})

    empty_label_files = 0
    object_count = 0
    for label_path in label_paths:
        class_ids, issues = parse_label_file(label_path, nc)
        label_issues.extend(issues)
        if not class_ids:
            empty_label_files += 1
        object_count += len(class_ids)
        class_counter.update((split, class_id) for class_id in class_ids if 0 <= class_id < nc)
        for class_id in set(class_ids):
            if 0 <= class_id < nc:
                image_class_counter.update([(split, class_id)])

    split_rows.append({
        "split": split,
        "images": len(image_paths),
        "labels": len(label_paths),
        "objects": object_count,
        "empty_label_files": empty_label_files,
        "missing_labels": len(missing_labels),
        "orphan_labels": len(orphan_labels),
    })

split_validation_df = pd.DataFrame(split_rows)
class_validation_df = pd.DataFrame([
    {
        "split": split,
        "class_id": class_id,
        "class_name": class_names[class_id],
        "annotations_boxes": class_counter[(split, class_id)],
        "images_with_class": image_class_counter[(split, class_id)],
    }
    for split in ["train", "val", "test"]
    for class_id in range(nc)
])

display(split_validation_df)
display(class_validation_df.pivot(index="class_name", columns="split", values="annotations_boxes").fillna(0).astype(int))
display(class_validation_df.pivot(index="class_name", columns="split", values="images_with_class").fillna(0).astype(int))

if pair_issues or label_issues:
    display(pd.DataFrame(pair_issues))
    display(pd.DataFrame(label_issues).head(50))
    raise ValueError(f"Dataset validation failed with {len(pair_issues)} pair issues and {len(label_issues)} label issues.")

print({"data_yaml": str(DATA_YAML), "nc": nc, "names": class_names, "validation": "passed"})

In [ ]:
UPGRADE_ULTRALYTICS = False

if UPGRADE_ULTRALYTICS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "ultralytics"])

from ultralytics import YOLO
import ultralytics

print({"ultralytics_version": ultralytics.__version__})

In [ ]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_yolo_model(primary_model: str, fallback_model: str | None = None) -> tuple[YOLO, str]:
    candidates = [primary_model]
    if fallback_model and fallback_model not in candidates:
        candidates.append(fallback_model)

    errors = {}
    for index, candidate in enumerate(candidates):
        try:
            model_instance = YOLO(candidate)
            if index > 0:
                print({"model_fallback_used": candidate, "primary_model_error": errors.get(primary_model)})
            return model_instance, candidate
        except Exception as error:  # noqa: BLE001
            errors[candidate] = str(error)
            if index == 0 and not ALLOW_MODEL_FALLBACK:
                raise RuntimeError(
                    f"Could not load {primary_model}. Set UPGRADE_ULTRALYTICS=True in the previous cell, "
                    "or choose an installed model weight."
                ) from error

    raise RuntimeError(f"Could not load any model candidate: {errors}")

set_seed(SEED)
model, SELECTED_MODEL_NAME = load_yolo_model(MODEL_NAME, FALLBACK_MODEL_NAME if ALLOW_MODEL_FALLBACK else None)
print({"selected_model_name": SELECTED_MODEL_NAME, "device": DEVICE})

In [ ]:
if RUN_SMOKE_TEST:
    smoke_model, smoke_model_name = load_yolo_model(MODEL_NAME, FALLBACK_MODEL_NAME if ALLOW_MODEL_FALLBACK else None)
    smoke_results = smoke_model.train(
        data=str(DATA_YAML),
        epochs=1,
        imgsz=640,
        batch=2,
        device=DEVICE,
        workers=0 if os.name == "nt" else 2,
        project=str(TRAINING_ROOT),
        name=f"{RUN_NAME}_smoke",
        exist_ok=True,
        plots=False,
    )
    print({"smoke_test_save_dir": str(smoke_results.save_dir), "model": smoke_model_name})
else:
    print("RUN_SMOKE_TEST is False; skipping one-epoch smoke test.")

In [ ]:
train_args = {
    "data": str(DATA_YAML),
    "epochs": EPOCHS,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "device": DEVICE,
    "workers": WORKERS,
    "optimizer": OPTIMIZER,
    "lr0": LR0,
    "lrf": LRF,
    "weight_decay": WEIGHT_DECAY,
    "cos_lr": COS_LR,
    "patience": PATIENCE,
    "save_period": SAVE_PERIOD,
    "seed": SEED,
    "deterministic": True,
    "amp": AMP,
    "cache": CACHE,
    "close_mosaic": CLOSE_MOSAIC,
    "project": str(TRAINING_ROOT),
    "name": RUN_NAME,
    "exist_ok": EXIST_OK,
    "plots": True,
    "verbose": True,
}
if FREEZE is not None:
    train_args["freeze"] = FREEZE

print(json.dumps(train_args, indent=2))

if RUN_FULL_TRAINING:
    train_results = model.train(**train_args)
    RUN_DIR = Path(train_results.save_dir)
    BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
    LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
    print({"run_dir": str(RUN_DIR), "best_weights": str(BEST_WEIGHTS), "last_weights": str(LAST_WEIGHTS)})
else:
    RUN_DIR = TRAINING_ROOT / RUN_NAME
    BEST_WEIGHTS = RUN_DIR / "weights" / "best.pt"
    LAST_WEIGHTS = RUN_DIR / "weights" / "last.pt"
    print("RUN_FULL_TRAINING is False; training skipped.")

In [ ]:
RESUME_FROM = None

if RESUME_FROM:
    resume_path = Path(RESUME_FROM)
    if not resume_path.exists():
        raise FileNotFoundError(resume_path)
    resume_model = YOLO(str(resume_path))
    resume_results = resume_model.train(resume=True)
    print({"resumed_run_dir": str(resume_results.save_dir)})
else:
    print("Set RESUME_FROM to a last.pt path to resume an interrupted run.")

In [ ]:
weights_for_eval = BEST_WEIGHTS if BEST_WEIGHTS.exists() else LAST_WEIGHTS
if not weights_for_eval.exists():
    raise FileNotFoundError(f"No trained weights found for evaluation: {weights_for_eval}")

eval_model = YOLO(str(weights_for_eval))
val_metrics = eval_model.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=True)
test_metrics = eval_model.val(data=str(DATA_YAML), split="test", imgsz=IMGSZ, batch=BATCH, device=DEVICE, plots=True)
print({"weights": str(weights_for_eval), "val_metrics": str(val_metrics), "test_metrics": str(test_metrics)})

In [ ]:
test_images_dir = SPLIT_ROOT / "images" / "test"
sample_images = sorted(path for path in test_images_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)[:12]

if not sample_images:
    raise FileNotFoundError(f"No test images found in {test_images_dir}")

prediction_results = eval_model.predict(
    source=[str(path) for path in sample_images],
    imgsz=IMGSZ,
    conf=PREDICT_CONF,
    max_det=PREDICT_MAX_DET,
    device=DEVICE,
    save=True,
    save_txt=True,
    project=str(TRAINING_ROOT),
    name=f"{RUN_NAME}_test_predictions",
    exist_ok=True,
)
prediction_dir = Path(prediction_results[0].save_dir)
print({"prediction_dir": str(prediction_dir), "sample_count": len(sample_images)})

for preview_path in sorted(prediction_dir.glob("*"))[:6]:
    if preview_path.suffix.lower() in IMAGE_EXTENSIONS:
        display(IPythonImage(filename=str(preview_path), width=700))

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
copied_best_path = MODELS_DIR / f"{RUN_NAME}_best_{timestamp}.pt"
shutil.copy2(weights_for_eval, copied_best_path)
print({"copied_best_weights": str(copied_best_path)})

if EXPORT_ONNX:
    try:
        exported_path = eval_model.export(format="onnx", imgsz=IMGSZ, dynamic=True, simplify=False)
        print({"onnx_export": str(exported_path)})
    except Exception as error:  # noqa: BLE001
        print({"onnx_export_error": str(error)})
else:
    print("EXPORT_ONNX is False; export skipped.")